# Palmer Penguins — Python Exploration

This notebook walks through the **Palmer Penguins** dataset using Python.  
The data were collected by Dr. Kristen Gorman at Palmer Station, Antarctica, and cover
morphometric measurements for three penguin species across three islands.

> **Reference:** Gorman KB, Williams TD, Fraser WR (2014). PLoS ONE 9(3):e90081.  
> **Package:** [`palmerpenguins`](https://allisonhorst.github.io/palmerpenguins/) · Python port by `mckinney`.

---

## 1. Setup

Check for required packages and install anything missing, then import.

In [1]:
import importlib, subprocess, sys

required = ["palmerpenguins", "pandas", "matplotlib", "seaborn", "statsmodels", "scipy"]
missing  = [p for p in required if importlib.util.find_spec(p) is None]

if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All packages present.")

All packages present.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from palmerpenguins import load_penguins
import statsmodels.formula.api as smf
from scipy import stats

sns.set_theme(style="ticks", palette="colorblind", font_scale=1.1)

PALETTE = {"Adelie": "#FF8C00", "Chinstrap": "#9400D3", "Gentoo": "#008B8B"}

## 2. Data Loading and Cleaning

The dataset has **344 observations** and **8 variables**: species, island, four morphometric
measurements, sex, and year. Eleven rows contain at least one missing value.

In [3]:
penguins = load_penguins()
print(f"Shape (raw):   {penguins.shape}")
print(f"Missing cells: {penguins.isna().sum().sum()}")

df = penguins.dropna().copy()
print(f"Shape (clean): {df.shape}")
df.head()

Shape (raw):   (344, 8)
Missing cells: 19
Shape (clean): (333, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007


In [4]:
numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df.groupby("species")[numeric_cols].mean().round(2)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
species,,,,
Adelie,38.82,18.35,190.10,3706.16
Chinstrap,48.83,18.42,195.82,3733.09
Gentoo,47.57,15.00,217.24,5092.44


---

## 3. Meet the Penguins

Before diving into visualisations, it helps to see what we are working with.
The artwork below by [Allison Horst](https://www.allisonhorst.com/) has become
iconic in the data-science teaching community.

- **Left** — the three species: *Adélie*, *Chinstrap*, and *Gentoo*
- **Centre** — hex sticker for the `palmerpenguins` package
- **Right** — diagram showing how **bill length** and **bill depth** are measured

> *Artwork by @allison_horst — CC BY 4.0. Please cite when reusing.*

---

## 3. Visualisations

### 3.1 Scatter Matrix (Pair Plot)

A pair plot gives a quick overview of pairwise relationships between all four numeric
measurements. Diagonal panels show per-species KDE curves; off-diagonal panels are
scatter plots coloured by species.

Notice the clear separation between Gentoo (teal) and the two smaller species in both
flipper length and body mass.

In [ ]:
g = sns.pairplot(
    df[numeric_cols + ["species"]],
    hue="species",
    palette=PALETTE,
    diag_kind="kde",
    plot_kws={"alpha": 0.55, "s": 25},
    diag_kws={"fill": True, "alpha": 0.3},
)
g.figure.suptitle("Pairwise morphometric relationships", y=1.01, fontsize=14)
plt.show()

### 3.2 Violin Plot — Flipper Length by Species

Violin plots combine a box plot with a mirrored KDE, giving a richer picture of the
distribution shape. The `split=True` option places male and female distributions
side-by-side within each violin, exposing sexual dimorphism at a glance.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(
    data=df,
    x="species", y="flipper_length_mm",
    hue="sex", split=True,
    palette={"male": "#4C72B0", "female": "#DD8452"},
    inner="quartile", linewidth=1.1,
    ax=ax,
)
ax.set_title("Flipper length by species and sex", fontsize=14)
ax.set_xlabel("Species")
ax.set_ylabel("Flipper length (mm)")
sns.despine()
plt.tight_layout()
plt.show()

### 3.3 Strip Plot with Means — Bill Length by Island

A strip plot shows every individual data point while overlaying the group mean.
The jitter along the x-axis prevents overplotting. On Biscoe island, only Adelie and
Gentoo are found; Dream mixes Adelie with Chinstrap; Torgersen is Adelie-only.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.stripplot(
    data=df,
    x="island", y="bill_length_mm",
    hue="species", palette=PALETTE,
    dodge=True, jitter=True, alpha=0.6, size=5,
    ax=ax,
)
# Overlay species means per island
means = df.groupby(["island", "species"])["bill_length_mm"].mean().reset_index()
sns.pointplot(
    data=means,
    x="island", y="bill_length_mm",
    hue="species", palette=PALETTE,
    dodge=0.55, join=False,
    markers="D", scale=0.9, errorbar=None,
    ax=ax, legend=False,
)
ax.set_title("Bill length by island and species (diamonds = means)", fontsize=13)
ax.set_xlabel("Island")
ax.set_ylabel("Bill length (mm)")
ax.legend(title="Species", loc="lower right")
sns.despine()
plt.tight_layout()
plt.show()

### 3.4 Correlation Heatmap

A Pearson correlation heatmap summarises linear associations between numeric
variables (computed on the full cleaned dataset, ignoring species). The strong
positive correlation between flipper length and body mass stands out, as does the
apparent negative correlation between bill depth and bill length — which reverses
within each species (Simpson's Paradox).

In [ ]:
corr = df[numeric_cols].corr()

mask = pd.DataFrame(False, index=corr.index, columns=corr.columns)
import numpy as np
mask.values[np.triu_indices_from(mask, k=1)] = True  # hide upper triangle

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    corr, mask=mask,
    annot=True, fmt=".2f",
    cmap="coolwarm", vmin=-1, vmax=1,
    linewidths=0.5, square=True,
    xticklabels=[c.replace("_mm", "").replace("_g", "").replace("_", " ") for c in numeric_cols],
    yticklabels=[c.replace("_mm", "").replace("_g", "").replace("_", " ") for c in numeric_cols],
    ax=ax,
)
ax.set_title("Pearson correlation — morphometric variables", fontsize=13)
plt.tight_layout()
plt.show()

---

## 4. Regression Analysis

We model **body mass** as a function of flipper length, bill dimensions, and species
using ordinary least squares. Using `statsmodels` lets us inspect a full regression
summary including coefficient estimates, standard errors, and diagnostic statistics.

$$
\text{body mass}_i = \beta_0
  + \beta_1 \cdot \text{flipper}
  + \beta_2 \cdot \text{bill length}
  + \beta_3 \cdot \text{bill depth}
  + \beta_4 \cdot \mathbf{1}[\text{Chinstrap}]
  + \beta_5 \cdot \mathbf{1}[\text{Gentoo}]
  + \varepsilon_i
$$

In [ ]:
formula = "body_mass_g ~ flipper_length_mm + bill_length_mm + bill_depth_mm + C(species)"
model = smf.ols(formula, data=df).fit()
print(model.summary())

### Coefficient Plot

Visualising the coefficient estimates with 95% confidence intervals makes it easy to
judge which predictors are precisely estimated and which cross zero.

In [ ]:
coef_df = (
    pd.DataFrame({
        "coef":  model.params,
        "lower": model.conf_int()[0],
        "upper": model.conf_int()[1],
    })
    .drop(index="Intercept")
    .rename(index=lambda s: (
        s.replace("C(species)[", "").replace("T.", "Species: ").replace("]", "")
         .replace("_mm", "").replace("_g", "").replace("_", " ")
    ))
    .sort_values("coef")
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(
    coef_df.index, coef_df["coef"],
    xerr=[coef_df["coef"] - coef_df["lower"], coef_df["upper"] - coef_df["coef"]],
    color=["#4C72B0" if v >= 0 else "#DD8452" for v in coef_df["coef"]],
    edgecolor="white", height=0.55, capsize=4,
)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("OLS coefficients with 95% CI\n(body mass outcome)", fontsize=13)
ax.set_xlabel("Coefficient estimate (g or g/mm)")
sns.despine()
plt.tight_layout()
plt.show()

print(f"\nAdj. R² = {model.rsquared_adj:.4f} | RMSE = {model.mse_resid**0.5:.1f} g | N = {int(model.nobs)}")

---

## 5. Summary

| Step | Details |
|------|---------|
| Packages | `palmerpenguins`, `pandas`, `seaborn`, `statsmodels`, `scipy` |
| Data | 333 complete observations after NA removal |
| Pair plot | Pairwise scatter + KDE diagonal |
| Violin plot | Flipper length by species × sex |
| Strip plot | Bill length by island, means overlaid |
| Heatmap | Pearson correlations of four morphometrics |
| Regression | OLS body mass ~ flipper + bill dims + species |
| Coef plot | Horizontal bar chart of estimates with 95% CI |